In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Build an agent with one function tool

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Agents in ADK

An agent in the [Agent Development Kit](https://google.github.io/adk-docs/) (ADK) has a name, a model, an instruction and a list of tools. The model reads the instruction and the user's message, decides whether a tool can help, and writes the reply.

### Function tools

A [function tool](https://google.github.io/adk-docs/tools/function-tools/) is an ordinary Python function. ADK reads its name, its parameters and its docstring, and describes the function to the model. When the model decides to call it, ADK runs the function and sends the result back to the model.

### The store stock tool

This quickstart uses `check_store_stock`, the same tool the Cymbal Beauty store agent uses. It reads the store inventory in BigQuery and returns what is on the shelf, what is in the backroom, and whether the product can be picked up in store.

<img width="60%" src="../../docs/diagrams/q01.png" alt="An agent with one function tool that reads store stock from BigQuery" />

### Objectives

In this tutorial, you will learn how to build the smallest useful ADK agent: one agent and one function tool.

You will complete the following tasks:

- Call the store stock tool directly and read what it returns
- Define an agent that uses the tool
- Run the agent and see the tool call in its events

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Gemini on Vertex AI pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing), [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded during setup, in your own namespace. Set your project ID and the namespace you chose.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# This notebook sits two folders below the repository root, where the shared store tools live
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the notebook output to the agent's own events: ADK marks experimental features with a
# UserWarning, SDKs announce renamed classes with a FutureWarning, and the Gen AI SDK logs a note
# whenever a response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

from google.adk.agents import LlmAgent
from google.adk.models import Gemini
from google.adk.runners import InMemoryRunner
from google.genai import types

from agents.cymbal_store_ops.tools.domain_tools import check_store_stock

### Choose the model

The agent in this tutorial uses Gemini 3.8 Flash. The retry options make the SDK retry a request that fails with a temporary error, such as a 429 or a 500, instead of failing the turn.

In [4]:
model = Gemini(
    model="gemini-3.8-flash",
    retry_options=types.HttpRetryOptions(attempts=4, initial_delay=2.0),
)

## Call the tool directly

A function tool is plain Python, so you can call it without a model. Check the Lumière Hydra Cream at the Cymbal Beauty store in Naperville:

In [5]:
check_store_stock(product_name="Hydra Cream", city="Naperville")

{'status': 'SUCCESS',
 'rows': [{'store_id': 'S-014',
   'store_name': 'Cymbal Beauty Naperville',
   'city': 'Naperville',
   'state': 'IL',
   'product_id': 'P-0101',
   'product_name': 'Lumière Hydra Cream',
   'category': 'skincare',
   'locked_case': False,
   'price_usd': 28.0,
   'on_hand': 7,
   'on_shelf_qty': 0,
   'backroom_qty': 7,
   'reorder_point': 12,
   'shelf_capacity': 18,
   'bopis_eligible': True,
   'updated_at': '2026-10-03T07:00-05:00',
   'is_osa_exception': True,
   'recommendation': 'backroom_check',
   'recommended_actions': ['backroom_check', 'replenish'],
   'replenishment_in_transit': False,
   'replenishment_delayed': True,
   'reason': 'nothing on the shelf while 7 unit(s) sit in the backroom; on_hand 7 is below the reorder point 12 and its inbound shipment is delayed, not in transit'}]}

One row comes back, for Cymbal Beauty Naperville (S-014): 7 units on hand, none on the shelf and all 7 in the backroom, below the reorder point of 12. The tool flags this as an on-shelf availability exception (`is_osa_exception`) and recommends a backroom check. `bopis_eligible: True` means guests can order it for pick-up.

The docstring and the parameter names are all the model knows about the tool, so they need to say what the tool does and what each argument means:

In [6]:
print(check_store_stock.__doc__)

Stock position of one product: on_hand, on_shelf_qty, backroom_qty, reorder_point, shelf_capacity, BOPIS
    eligibility, and the OSA recommendation (backroom_check | replenish | cycle_count | escalate | none).

    Args:
        product_name: product name or a distinctive part of it (e.g. "Hydra Cream"), or a product id.
        city: look at the Cymbal Beauty store(s) in this city instead of the signed-in store (e.g. "Naperville").
        store_id: look at this store instead of the signed-in store (district managers).
    


## Define the agent

The instruction tells the model what job it has and how to answer. The tool list gives it one tool.

In [7]:
instruction = """You help Cymbal Beauty store associates answer one question: is a product in stock at our
stores in a given city?

Call check_store_stock with the product name and the city. Answer in one or two sentences: store name,
units on hand with how many are on the shelf and in the backroom, and whether pick-up is available.
If the tool returns an error, say what could not be checked. Never guess a quantity."""

agent = LlmAgent(
    name="hello_tool_agent",
    model=model,
    description="Answers whether a Cymbal Beauty product is in stock at stores in a city.",
    instruction=instruction,
    tools=[check_store_stock],
)

## Run the agent

An ADK `Runner` sends messages to the agent and returns a stream of events. `InMemoryRunner` keeps the conversation in memory, which is enough for a notebook. Create a runner and a session:

In [8]:
runner = InMemoryRunner(agent=agent, app_name="hello_tool_agent")
session = await runner.session_service.create_session(app_name="hello_tool_agent", user_id="associate")

Define a helper that sends one message and prints what happened: each tool call with its arguments, and the agent's reply.

In [9]:
async def ask(question: str) -> None:
    """Send one message to the agent and print its tool calls and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            print(f"[{event.author}] calls {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n{text}\n")

Ask about a product. The model calls `check_store_stock` once and answers from the result:

In [10]:
await ask("Is Lumière Hydra Cream in stock in Naperville?")

[hello_tool_agent] calls check_store_stock({'city': 'Naperville', 'product_name': 'Lumière Hydra Cream'})



At Cymbal Beauty Naperville, Lumière Hydra Cream has 7 units on hand, with 0 on the shelf and 7 in the backroom. Store pick-up is available.



The agent made one tool call and answered from its result: 7 on hand, 0 on the shelf, 7 in the backroom, pick-up available. The wording differs from run to run; the tool call and the numbers should match.

Ask about another product. The agent calls the same tool with a different product name:

In [11]:
await ask("Can a guest pick up Noir Velvet Eau de Parfum in Naperville today?")

[hello_tool_agent] calls check_store_stock({'city': 'Naperville', 'product_name': 'Noir Velvet Eau de Parfum'})



At Cymbal Beauty Naperville, Noir Velvet Eau de Parfum has 5 units on hand (2 on the shelf and 3 in the backroom). Store pick-up is available today.



Noir Velvet Eau de Parfum has 5 units at Naperville: 2 on the shelf and 3 in the backroom. Try your own questions too, for example:

```
Do we have Velvet Styling Cream in Naperville?
```

## Run the agent in the ADK developer UI

`agent.py` in this folder defines the same agent. The developer UI needs Python package names, so first copy the quickstarts into `build/quickstart_apps` under names it accepts, then start the UI from the repository root:

```bash
uv run python scripts/quickstart_apps.py 01-hello-tool-agent
uv run adk web build/quickstart_apps --port 8001
```

Open http://localhost:8001 and choose `qs_01_hello_tool_agent`.

## Cleaning up

This notebook creates no cloud resources. The session lived in memory and ends when you restart the kernel.

## What's next

- [Function tools in ADK](https://google.github.io/adk-docs/tools/function-tools/)
- [Quickstart 02: answer questions from store procedures with Vertex AI Search](../02-rag-knowledge-agent/walkthrough.ipynb)